```sql

Goal:
- simulate "online" scoring over a transaction stream
- compute velocity features incrementally (past-only)
- compare offline vs online feature values on the same rows (parity check)

Why:
If features differ between offline training and online serving, model quality will silently degrade

```

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

In [2]:
from sklearn.metrics import roc_auc_score, average_precision_score
from lightgbm import LGBMClassifier

#### Import data

In [3]:
train_df=pd.read_pickle('../data/interim/train_joined_split.pkl')

In [4]:
valid_df=pd.read_pickle('../data/interim/valid_joined_split.pkl')

In [5]:
train_df.shape, valid_df.shape

((472432, 434), (118108, 434))

In [6]:
#Sorting by Txn DT
train_df = train_df.sort_values("TransactionDT").reset_index(drop=True)
valid_df = valid_df.sort_values("TransactionDT").reset_index(drop=True)

In [7]:
base_features=["TransactionAmt",
               "ProductCD",
               "card1", "card2", "card3", "card4", "card5", "card6",
               "addr1", "addr2",
               "P_emaildomain", "R_emaildomain",
               "DeviceType", "DeviceInfo"
              ]
target_col="isFraud"

In [8]:
available_features = [c for c in base_features if c in train_df.columns]

In [9]:
# Adding some transformation on features
for df_ in [train_df,valid_df]:
    df_["log_txnamt"]=np.log1p(df_["TransactionAmt"]) #log1p works best for skewed and zero values well
    df_["has_identity"]=(df_[["DeviceType", "DeviceInfo"]].notna().any(axis=1).astype(int)) #Any one value is available

In [10]:
new_features = ["log_txnamt", "has_identity"]

In [12]:
feature_cols=available_features+new_features